<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_Final_Notebook_Config.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [2]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 99% 92.0M/92.7M [00:06<00:00, 17.9MB/s]
100% 92.7M/92.7M [00:06<00:00, 14.6MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [3]:
import wandb

wandb.login(key = userdata.get("WB"))

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ratludu (ratludu-backpack-S5E2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import kagglehub

# Download latest version
path1 = kagglehub.dataset_download("souradippal/student-bag-price-prediction-dataset")

print("Path to dataset files:", path1)

100%|██████████| 1.23M/1.23M [00:01<00:00, 1.27MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/souradippal/student-bag-price-prediction-dataset/versions/1


In [5]:
!pip install dask-cuda==24.12.0

In [6]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 586, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 586 (delta 122), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (586/586), 191.99 KiB | 19.20 MiB/s, done.
Resolving deltas: 100% (296/296), done.
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 8.14s
 Downloaded datashader
 Downloaded cuspatial-cu12
 Downloaded cucim-cu12
 Downloaded scikit-image
 Downloaded libcuspatial-cu12
 Downloaded cugraph-cu12
Prepared 11 packages in 13.93s
Uninstalled 1 package in 65ms
Installed 11 packages in 13ms
 + cucim-cu12==24.12.0
 + cugraph-cu12==24.12.0
 + cuproj-cu12==24.12.0
 + cuspatial-cu12==24.12.0
 + cuxfilter-cu12==24.12.0
 + datashader==0.17.0
 + jupyter-server-proxy==4.4.0
 + libcuspatial-cu12==24.12.0
 + pyct==0.5.0
 - scikit-image==0.25.2
 + scikit-image==0.24.0
 + sim

In [7]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.4 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

KeyboardInterrupt: 

In [ ]:
# Initialise Project

wandb.init(
    # set the wandb entity where your project will be logged (generally your team name)
    entity="ratludu-backpack-S5E2",

    # set the wandb project where this run will be logged
    project="Catboost-Backpack",

    # track hyperparameters and run metadata
    config={
            'learning_rate': 0.02,
            #'l2_leaf_reg':5,
            'per_float_feature_quantization':'8:border_count=1024',
            'iterations': 5_000,
            'task_type': "GPU",
            'grow_policy': 'Lossguide',
            'random_state': 42,
            #'early_stopping_rounds':250,
            'verbose': 250,
            'loss_function':'RMSE'
    }
)

In [ ]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [ ]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [ ]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [ ]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)
original = pd.read_csv(path1+"/Noisy_Student_Bag_Price_Prediction_Dataset.csv")
original.dropna(subset=['Price'],inplace = True)

In [ ]:
train = pd.concat([train,train_ex, original], axis = 0, ignore_index = True)

In [ ]:
#train = train.sample(frac = 0.1, random_state = 42, ignore_index = True)

In [ ]:
COMBO = []
for i,c in enumerate(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment','Waterproof', 'Style', 'Color']):
    #print(f"{c}, ",end="")
    combine = pd.concat([train[c],test[c]],axis=0)
    combine,_ = pd.factorize(combine)
    train[c] = combine[:len(train)]
    test[c] = combine[len(train):]
    n = f"{c}_wc"
    train[n] = train[c]*100 + train["Weight Capacity (kg)"]
    test[n] = test[c]*100 + test["Weight Capacity (kg)"]
    COMBO.append(n)
print()
print(f"We engineer {len(COMBO)} new columns!")
print( COMBO )

In [ ]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand_wc', 'Material_wc', 'Size_wc', 'Compartments_wc', 'Laptop Compartment_wc', 'Waterproof_wc', 'Style_wc', 'Color_wc']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
cat3_oof = np.zeros(len(train))
cat3_preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features+['Price']].reset_index(drop=True).copy(), train.loc[test_idx, features].reset_index(drop=True).copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])**2
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])**2
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])**2

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    x_train['Material_TE-Color_TE'] = x_train['Material_TE']*x_train['Color_TE']
    x_val['Material_TE-Color_TE'] = x_val['Material_TE']*x_val['Color_TE']
    x_test['Material_TE-Color_TE'] = x_test['Material_TE']*x_test['Color_TE']

    x_train['Sizexbrand-Materialxweight'] = (x_train['Sizexbrand']-x_train['Materialxweight'])**2
    x_val['Sizexbrand-Materialxweight'] = (x_val['Sizexbrand']-x_val['Materialxweight'])**2
    x_test['Sizexbrand-Materialxweight'] = (x_test['Sizexbrand']-x_test['Materialxweight'])**2

    x_train['Sizexweight-Laptop Compartment_TE'] = x_train['Sizexweight']/x_train['Laptop Compartment_TE']
    x_val['Sizexweight-Laptop Compartment_TE'] = x_val['Sizexweight']/x_val['Laptop Compartment_TE']
    x_test['Sizexweight-Laptop Compartment_TE'] = x_test['Sizexweight']/x_test['Laptop Compartment_TE']

    x_train['Color_TE-Waterproof_TE'] = x_train['Color_TE']**2+x_train['Waterproof_TE']*x_train['Color_TE']
    x_val['Color_TE-Waterproof_TE'] = x_val['Color_TE']**2+x_val['Waterproof_TE']*x_val['Color_TE']
    x_test['Color_TE-Waterproof_TE'] = x_test['Color_TE']**2+x_test['Waterproof_TE']*x_test['Color_TE']

    x_train['Price/Weight'] = x_train['Price']/x_train['Weight Capacity (kg)']

    x_val['Price'] = y_val
    x_val['Price/Weight'] = x_val['Price']/x_val['Weight Capacity (kg)']
    x_val.drop(['Price'], axis = 1, inplace = True)

    features2 = list(x_val.columns)
    kf2 = KFold(n_splits=5, shuffle=True, random_state=42)
    for j, (train_index2, test_index2) in enumerate(kf2.split(x_train)):
        print(f" ## INNER Fold {j+1} (outer fold {fold+1}) ##")

        x_train2 = x_train.loc[train_index2,features2+['Price']].copy()
        x_valid2 = x_train.loc[test_index2,features2].copy()
        for col in drop:
          aggs = {}

          aggs['Price'] = ["std","median","min","max","skew"]

          train_agg = x_train2.groupby([col]).agg(aggs)
          train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
          col_names = {s:col+"_"+s+"_aggs_price" for s in ["std","median","min","max","skew"]}
          train_agg.rename(columns = col_names, inplace = True)

          x_valid2 = x_valid2.merge(train_agg, how='left', on = col)

          for c in list(col_names.values()):
            x_train.loc[test_index2,c] = x_valid2[c].values

        for col in drop:
          aggs = {}

          aggs['Price/Weight'] = ["mean","std","median","min","max","skew"]

          train_agg = x_train2.groupby([col]).agg(aggs)
          train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
          col_names = {s:col+"_"+s+"_aggs_price_weight" for s in ["mean","std","median","min","max","skew"]}
          train_agg.rename(columns = col_names, inplace = True)

          x_valid2 = x_valid2.merge(train_agg, how='left', on = col)

          for c in list(col_names.values()):
            x_train.loc[test_index2,c] = x_valid2[c].values


    for col in drop:
      aggs = {}

      aggs['Price'] = ["std","median","min","max","skew"]

      train_agg = x_train.groupby([col]).agg(aggs)
      train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
      col_names = {s:col+"_"+s+"_aggs_price" for s in ["std","median","min","max","skew"]}
      train_agg.rename(columns = col_names, inplace = True)

      x_val = x_val.merge(train_agg, how='left', on = col)
      x_test = x_test.merge(train_agg, how='left', on = col)

    for col in drop:
      aggs = {}

      aggs['Price/Weight'] = ["mean","std","median","min","max","skew"]

      train_agg = x_train.groupby([col]).agg(aggs)
      train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
      col_names = {s:col+"_"+s+"_aggs_price_weight" for s in ["mean","std","median","min","max","skew"]}
      train_agg.rename(columns = col_names, inplace = True)

      x_val = x_val.merge(train_agg, how='left', on = col)
      x_test = x_test.merge(train_agg, how='left', on = col)

    for col in drop:

      if col == 'Weight Capacity (kg)':
        pass

      else:
        aggs = {}

        aggs['Weight Capacity (kg)'] = ["mean","std","median","min","max","skew"]

        train_agg = x_train.groupby([col]).agg(aggs)
        train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
        col_names = {s:col+"_"+s+"_aggs_weight" for s in ["mean","std","median","min","max","skew"]}
        train_agg.rename(columns = col_names, inplace = True)

        x_train = x_train.merge(train_agg, how='left', on = col)
        x_val = x_val.merge(train_agg, how='left', on = col)
        x_test = x_test.merge(train_agg, how='left', on = col)


    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    x_train.drop(['Price'], axis = 1, inplace = True)

    x_train.drop(['Price/Weight'], axis = 1, inplace = True)
    x_val.drop(['Price/Weight'], axis = 1, inplace = True)

    print(x_train.columns)

    model = CatBoostRegressor(**wandb.config, cat_features = cats)

    model.fit(x_train, y_train, eval_set=(x_val,y_val))

    val_preds = model.predict(x_val)

    cat3_oof[test_idx] = val_preds

    cat3_preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')
    wandb.log({"RMSE": score})

print(f"The average CV is {np.average(m)}")
wandb.log({"Average_Score":np.average(m)})
wandb.finish()

In [ ]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = cat3_preds
submission.to_csv("submission.csv", index = False)

submission

In [ ]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission'

In [ ]:
!kaggle competitions submissions -c {competition}

In [ ]:
from google.colab import runtime
runtime.unassign()